# Construction-site PPE and worker detection with YOLO

Mini-project: fine-tune a pretrained YOLO26 detector to find workers, helmets, missing helmets and forklifts on construction sites, then use the detections for a simple worker-forklift proximity check (a first step toward struck-by hazard monitoring).

**Run this on a GPU.** Kaggle Notebooks (Settings > Accelerator > GPU) or any cloud GPU works. On a laptop CPU, training takes many hours.

## 1. Setup

In [1]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()   # prints Python, torch and GPU info

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
Setup complete ✅ (4 CPUs, 31.3 GB RAM, 5642.4/8156.9 GB disk)


## 2. Train

- `yolo26n.pt` is a small model already trained on everyday objects (COCO). We *fine-tune* it on construction images.
- The **Construction-PPE** dataset (Ultralytics, CC BY 4.0) downloads automatically the first time: 800 training and 200 validation images, 12 classes.
- `epochs=50` means the model sees every training image 50 times.

In [2]:
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
results = model.train(
    data="construction-ppe.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    patience=15,        # stop early if validation stops improving
    name="ppe_yolo26n",
    seed=0,
)
BEST = str(model.trainer.best)        # path to the best weights
RUN_DIR = str(model.trainer.save_dir) # folder with curves and plots
print("Best weights:", BEST)
print("Run folder:  ", RUN_DIR)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=construction-ppe.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=ppe_yolo26n, nbs=64, nms=None, opset=None, optimize=False, optimize

/usr/local/lib/python3.13/dist-packages/ray/train/_internal/session.py:683: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


       2/50      3.24G      1.465      2.803   0.006099        310        640: 100% ━━━━━━━━━━━━ 50/50 5.2it/s 9.7s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 3.6it/s 2.0s0.4s
                   all        200       2961      0.642      0.505      0.509      0.341

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       3/50      3.24G      1.424      1.991   0.005764        434        640: 100% ━━━━━━━━━━━━ 50/50 5.3it/s 9.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 4.1it/s 1.7s0.3s
                   all        200       2961      0.784      0.543      0.593      0.394

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
       4/50      3.24G      1.365      1.618   0.005817        386        640: 100% ━━━━━━━━━━━━ 50/50 5.3it/s 9.5s0.2s
                 Class     Images  Instances  

## 3. Evaluate

Key numbers to report:
- **Precision (P):** of the boxes the model drew, how many were correct.
- **Recall (R):** of the real objects, how many the model found.
- **mAP50:** overall detection quality (higher is better, max 1.0).

In [3]:
from ultralytics import YOLO
best = YOLO(BEST)
metrics = best.val(data="construction-ppe.yaml", split="val")

import pandas as pd
names = best.names
rows = []
for i, c in enumerate(metrics.box.ap_class_index):
    p, r, ap50, ap = metrics.box.class_result(i)
    rows.append({"class": names[int(c)], "precision": round(p, 3), "recall": round(r, 3),
                 "mAP50": round(ap50, 3), "mAP50-95": round(ap, 3)})
table = pd.DataFrame(rows).sort_values("mAP50", ascending=False)
print(f"Overall mAP50 = {metrics.box.map50:.3f}, mAP50-95 = {metrics.box.map:.3f}")
table.to_csv("results_per_class.csv", index=False)
table

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14912MiB)
YOLO26n summary (fused): 121 layers, 2,392,408 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 5474.2±1155.4 MB/s, size: 852.4 KB)
val: Scanning /kaggle/working/datasets/construction-ppe/labels/val.cache... 200 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 200/200 64.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 3.7it/s 3.5s0.2s
                   all        200       2961      0.849      0.683      0.746      0.534
                helmet        146        294      0.917      0.794      0.843      0.642
                gloves        136        381      0.889      0.737      0.815      0.606
                  vest        157        340      0.938      0.838      0.904      0.704
                 boots        155        481      0.807      0.694       0.78      0.578
               g

,class,precision,recall,mAP50,mAP50-95
5,Person,0.948,0.898,0.947,0.785
4,goggles,0.885,0.935,0.947,0.615
2,vest,0.938,0.838,0.904,0.704
0,helmet,0.917,0.794,0.843,0.642
10,forklift,0.811,0.836,0.841,0.598
1,gloves,0.889,0.737,0.815,0.606
3,boots,0.807,0.694,0.780,0.578
9,no_boots,0.876,0.597,0.723,0.563
11,no_vest,0.893,0.643,0.700,0.481
8,no_gloves,0.867,0.542,0.635,0.393


## 4. See it work

Run the model on validation images and save pictures with boxes drawn. Use 3-4 good ones (and one honest failure) in your README.

In [4]:
import glob, os
from ultralytics.data.utils import check_det_dataset
val_dir = check_det_dataset("construction-ppe.yaml")["val"]   # finds where the dataset was saved
val_imgs = sorted(glob.glob(os.path.join(val_dir, "*.jpg")) + glob.glob(os.path.join(val_dir, "*.png")))[:12]
print(len(val_imgs), "images from", val_dir)
preds = best.predict(val_imgs, conf=0.35, save=True, name="examples", exist_ok=True)
print("Pictures saved to:", preds[0].save_dir)


12 images from /kaggle/working/datasets/construction-ppe/images/val

0: 640x640 1 boots, 1 Person, 2 no_glovess, 1 no_boots, 1 no_vest, 5.9ms
1: 640x640 7 Persons, 5.9ms
2: 640x640 4 helmets, 4 glovess, 3 vests, 5 bootss, 3 Persons, 1 no_gloves, 5.9ms
3: 640x640 4 helmets, 3 glovess, 4 vests, 9 bootss, 4 Persons, 5.9ms
4: 640x640 1 helmet, 1 vest, 5 Persons, 1 no_helmet, 1 forklift, 5.9ms
5: 640x640 1 vest, 2 bootss, 1 Person, 1 no_goggle, 1 no_gloves, 1 forklift, 5.9ms
6: 640x640 1 helmet, 2 vests, 3 Persons, 1 no_boots, 1 no_vest, 5.9ms
7: 640x640 2 helmets, 4 vests, 10 bootss, 4 Persons, 1 no_helmet, 1 no_gloves, 5.9ms
8: 640x640 4 vests, 5 Persons, 5 forklifts, 5.9ms
9: 640x640 4 helmets, 3 vests, 4 Persons, 1 no_helmet, 1 no_boots, 5.9ms
10: 640x640 2 helmets, 5 vests, 1 boots, 7 Persons, 5.9ms
11: 640x640 2 helmets, 2 glovess, 2 vests, 6 bootss, 3 Persons, 1 forklift, 1 no_vest, 5.9ms
Speed: 1.9ms preprocess, 5.9ms inference, 0.9ms postprocess per image at shape (1, 3, 640, 640)


## 5. Toward struck-by monitoring: worker-forklift proximity

A simple rule: flag any person whose box centre is close to a forklift box centre.

**Limitation (say this in your README):** distance here is in *pixels*, not metres. Real distances need camera calibration or depth sensing, which is exactly the problem a crane-safety project must solve.

In [5]:
import math

def proximity_alerts(result, max_px=150):
    names = result.names
    boxes = result.boxes
    people, forklifts = [], []
    for xyxy, cls in zip(boxes.xyxy.tolist(), boxes.cls.tolist()):
        cx, cy = (xyxy[0] + xyxy[2]) / 2, (xyxy[1] + xyxy[3]) / 2
        label = names[int(cls)]
        if label == "Person":
            people.append((cx, cy))
        elif label == "forklift":
            forklifts.append((cx, cy))
    alerts = []
    for p in people:
        for f in forklifts:
            d = math.dist(p, f)
            if d < max_px:
                alerts.append(round(d))
    return alerts

flagged = 0
for r in best.predict(val_imgs, conf=0.35, verbose=False):
    a = proximity_alerts(r)
    if a:
        flagged += 1
        print(os.path.basename(r.path), "-> person near forklift, pixel distances:", a)
print(f"{flagged} of {len(val_imgs)} images flagged")

source-223442b1ecf2d156710f2b828bb05db2.jpg -> person near forklift, pixel distances: [25]
source-3dec6deaf0d256258b125015a036bb67.jpg -> person near forklift, pixel distances: [145, 66, 140]
2 of 12 images flagged


## 6. Optional: tracking on a video

Tracking gives each worker a stable ID across frames, which later lets you predict where they are walking. Put any construction-site video (for example a free stock clip) next to this notebook.

In [ ]:
# best.track("site_video.mp4", tracker="bytetrack.yaml", conf=0.35, save=True, name="tracking")

## 7. Save results for GitHub

Copy these into your repository's `results/` folder:
- `results.png` (training curves) from the run folder printed in step 2
- `confusion_matrix.png` from the same folder
- `results_per_class.csv`
- 3-4 images from the examples folder printed in step 4

Do **not** upload the dataset itself; link to it instead.

In [6]:
import shutil, os
os.makedirs("results", exist_ok=True)
shutil.copy(os.path.join(RUN_DIR, "results.png"), "results/")
shutil.copy(os.path.join(RUN_DIR, "confusion_matrix.png"), "results/")
shutil.copy("results_per_class.csv", "results/")
ex_dir = preds[0].save_dir
for i, f in enumerate(sorted(os.listdir(ex_dir))[:6], 1):
    shutil.copy(os.path.join(ex_dir, f), f"results/example_{i}.jpg")
shutil.make_archive("results", "zip", "results")
print("Done:", os.listdir("results"))

Done: ['example_6.jpg', 'example_3.jpg', 'results_per_class.csv', 'example_1.jpg', 'example_2.jpg', 'results.png', 'example_5.jpg', 'confusion_matrix.png', 'example_4.jpg']
